In [61]:
# Import libraries
import torch
import numpy as np
import pandas as pd
import random

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print("Environment Ready")

Environment Ready


In [62]:
from sklearn.datasets import load_breast_cancer

# Load dataset
data = load_breast_cancer()

X = data.data
y = data.target

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (569, 30)
Shape of y: (569,)


In [63]:
df = pd.DataFrame(X, columns=data.feature_names)
df['target'] = y

df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [64]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness error         5

In [65]:
df['target'].value_counts() #0 - malignant, 1 - benign

target
1    357
0    212
Name: count, dtype: int64

In [66]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (455, 30)
Test shape: (114, 30)


In [67]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training data
X_train = scaler.fit_transform(X_train)

# Apply to test data
X_test = scaler.transform(X_test)

In [68]:
import torch

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [69]:
import torch.nn as nn
import torch.nn.functional as F

class BreastCancerMLP(nn.Module):
    def __init__(self):
        super(BreastCancerMLP, self).__init__()
        
        # Layers
        self.fc1 = nn.Linear(30, 64)
        self.fc2 = nn.Linear(64, 128)
        self.dropout1 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(128, 64)
        self.dropout2 = nn.Dropout(0.3)
        self.fc4 = nn.Linear(64, 2)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.dropout1(x)
        x = F.relu(self.fc3(x))
        x = self.dropout2(x)
        x = self.fc4(x)  # No softmax here (important)
        return x

In [70]:
model = BreastCancerMLP()
print(model)

BreastCancerMLP(
  (fc1): Linear(in_features=30, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=128, bias=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=128, out_features=64, bias=True)
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc4): Linear(in_features=64, out_features=2, bias=True)
)


In [71]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [72]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [73]:
def train_model(model, train_loader, criterion, optimizer, epochs=50):
    model.train()
    
    for epoch in range(epochs):
        running_loss = 0.0
        
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss:.4f}")

In [74]:
from sklearn.metrics import accuracy_score

def evaluate_model(model, test_loader):
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.numpy())
            all_labels.extend(y_batch.numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    print(f"Test Accuracy: {acc:.4f}")

In [75]:
train_model(model, train_loader, criterion, optimizer, epochs=50)

Epoch [1/50], Loss: 9.5093
Epoch [2/50], Loss: 5.3547
Epoch [3/50], Loss: 2.3342
Epoch [4/50], Loss: 1.3052
Epoch [5/50], Loss: 0.9274
Epoch [6/50], Loss: 0.7780
Epoch [7/50], Loss: 0.6197
Epoch [8/50], Loss: 0.6089
Epoch [9/50], Loss: 0.4679
Epoch [10/50], Loss: 0.5467
Epoch [11/50], Loss: 0.4815
Epoch [12/50], Loss: 0.3962
Epoch [13/50], Loss: 0.3523
Epoch [14/50], Loss: 0.3259
Epoch [15/50], Loss: 0.3921
Epoch [16/50], Loss: 0.2087
Epoch [17/50], Loss: 0.2389
Epoch [18/50], Loss: 0.1508
Epoch [19/50], Loss: 0.1522
Epoch [20/50], Loss: 0.1231
Epoch [21/50], Loss: 0.1847
Epoch [22/50], Loss: 0.2479
Epoch [23/50], Loss: 0.0965
Epoch [24/50], Loss: 0.0869
Epoch [25/50], Loss: 0.0762
Epoch [26/50], Loss: 0.0710
Epoch [27/50], Loss: 0.0252
Epoch [28/50], Loss: 0.0375
Epoch [29/50], Loss: 0.1152
Epoch [30/50], Loss: 0.0627
Epoch [31/50], Loss: 0.1332
Epoch [32/50], Loss: 0.0757
Epoch [33/50], Loss: 0.1117
Epoch [34/50], Loss: 0.0342
Epoch [35/50], Loss: 0.0235
Epoch [36/50], Loss: 0.0153
E

In [76]:
evaluate_model(model, test_loader)

Test Accuracy: 0.9561


In [77]:
from collections import defaultdict

def partition_non_iid(X, y, n_clients=5, alpha=0.5):
    """
    Splits dataset into non-IID partitions using Dirichlet distribution
    """
    
    n_classes = len(np.unique(y))
    client_data = {i: [] for i in range(n_clients)}
    
    # Get indices for each class
    class_indices = [np.where(y == i)[0] for i in range(n_classes)]
    
    for c in range(n_classes):
        np.random.shuffle(class_indices[c])

        # Dirichlet distribution
        proportions = np.random.dirichlet(np.repeat(alpha, n_clients))
        
        # Normalize proportions
        proportions = (np.cumsum(proportions) * len(class_indices[c])).astype(int)[:-1]
        
        split_indices = np.split(class_indices[c], proportions)
        
        for i, idx in enumerate(split_indices):
            client_data[i].extend(idx)

    # Ensure every client has at least one sample to avoid empty DataLoader issues
    empty_clients = [cid for cid, idx in client_data.items() if len(idx) == 0]
    for empty_cid in empty_clients:
        donor_cid = max(client_data, key=lambda k: len(client_data[k]))
        if len(client_data[donor_cid]) <= 1:
            continue
        client_data[empty_cid].append(client_data[donor_cid].pop())
    
    return client_data

In [78]:
client_partitions = partition_non_iid(X_train, y_train, n_clients=5, alpha=0.5)

In [79]:
client_loaders = {}

for client_id, indices in client_partitions.items():
    X_client = X_train[indices]
    y_client = y_train[indices]
    
    X_tensor = torch.tensor(X_client, dtype=torch.float32)
    y_tensor = torch.tensor(y_client, dtype=torch.long)
    
    dataset = TensorDataset(X_tensor, y_tensor)
    loader = DataLoader(dataset, batch_size=32, shuffle=True)
    
    client_loaders[client_id] = loader

In [80]:
for client_id, indices in client_partitions.items():
    y_client = y_train[indices]
    
    unique, counts = np.unique(y_client, return_counts=True)
    
    print(f"Client {client_id}:")
    for u, c in zip(unique, counts):
        print(f"  Class {u}: {c}")

Client 0:
  Class 0: 69
  Class 1: 4
Client 1:
  Class 0: 2
  Class 1: 2
Client 2:
  Class 0: 15
  Class 1: 28
Client 3:
  Class 0: 83
  Class 1: 216
Client 4:
  Class 0: 1
  Class 1: 35


In [81]:
import flwr as fl
from collections import OrderedDict
from flwr.common import ndarrays_to_parameters, parameters_to_ndarrays


In [82]:
class FlowerClient(fl.client.NumPyClient):
    def __init__(self, train_loader, local_epochs=3):
        self.model = BreastCancerMLP()
        self.train_loader = train_loader
        self.local_epochs = local_epochs

    def get_parameters(self, config):
        return [val.detach().cpu().numpy() for _, val in self.model.state_dict().items()]

    def set_parameters(self, parameters):
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.train()

        optimizer = torch.optim.Adam(self.model.parameters(), lr=0.001)
        criterion = torch.nn.CrossEntropyLoss()

        for _ in range(self.local_epochs):
            for X_batch, y_batch in self.train_loader:
                optimizer.zero_grad()
                outputs = self.model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()

        return self.get_parameters(config), len(self.train_loader.dataset), {}

    def evaluate(self, parameters, config):
        # Client-side evaluation uses local client data only; test set remains centralized.
        self.set_parameters(parameters)
        self.model.eval()

        criterion = torch.nn.CrossEntropyLoss()
        loss_sum, correct, total = 0.0, 0, 0

        with torch.no_grad():
            for X_batch, y_batch in self.train_loader:
                outputs = self.model(X_batch)
                loss = criterion(outputs, y_batch)
                _, predicted = torch.max(outputs, 1)

                batch_size = y_batch.size(0)
                loss_sum += loss.item() * batch_size
                total += batch_size
                correct += (predicted == y_batch).sum().item()

        avg_loss = loss_sum / total
        accuracy = correct / total
        return float(avg_loss), total, {"accuracy": float(accuracy)}

In [83]:
# Flower simulation configuration
NUM_CLIENTS = 5
LOCAL_EPOCHS = 3
NUM_ROUNDS = 20

server_model = BreastCancerMLP()

def client_fn(cid: str):
    train_loader = client_loaders[int(cid)]
    return FlowerClient(train_loader=train_loader, local_epochs=LOCAL_EPOCHS).to_client()

In [84]:
from collections import OrderedDict
from flwr.common import parameters_to_ndarrays

def evaluate_centralized(server_round, parameters, config):
    
    # ✅ Handle both cases (Parameters or list)
    if isinstance(parameters, list):
        ndarrays = parameters
    else:
        ndarrays = parameters_to_ndarrays(parameters)

    # Load model
    state_dict = OrderedDict(
        {k: torch.tensor(v) for k, v in zip(server_model.state_dict().keys(), ndarrays)}
    )
    server_model.load_state_dict(state_dict, strict=True)
    server_model.eval()

    criterion = torch.nn.CrossEntropyLoss()
    loss_sum, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = server_model(X_batch)
            loss = criterion(outputs, y_batch)
            _, preds = torch.max(outputs, 1)

            batch_size = y_batch.size(0)
            loss_sum += loss.item() * batch_size
            total += batch_size
            correct += (preds == y_batch).sum().item()

    avg_loss = loss_sum / total
    accuracy = correct / total

    print(f"Round {server_round}: centralized test accuracy = {accuracy:.4f}")

    return float(avg_loss), {"accuracy": float(accuracy)}

In [85]:
strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0,
    fraction_evaluate=0.0,
    min_fit_clients=3,
    min_available_clients=5,
    evaluate_fn=evaluate_centralized,
    initial_parameters=ndarrays_to_parameters(
        [val.detach().cpu().numpy() for _, val in server_model.state_dict().items()]
    ),
)

In [86]:
simulation_ran = False
history = None

try:
    history = fl.simulation.start_simulation(
        client_fn=client_fn,
        num_clients=NUM_CLIENTS,
        config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
        strategy=strategy,
        client_resources={"num_cpus": 1},
    )
    simulation_ran = True
    print("Flower simulation completed successfully.")
except Exception as e:
    print(f"Flower simulation failed: {e}")
    print("Compatibility note: install a Python 3.12-compatible Ray build, e.g. pip install 'ray[default]>=2.20.0'.")
    print("Alternative: run Flower start_server/start_numpy_client in separate processes without simulation.")

INFO flwr 2026-04-21 09:33:27,538 | app.py:178 | Starting Flower simulation, config: ServerConfig(num_rounds=20, round_timeout=None)
2026-04-21 09:33:38,644	INFO worker.py:1621 -- Started a local Ray instance.
INFO flwr 2026-04-21 09:33:56,232 | app.py:213 | Flower VCE: Ray initialized with resources: {'node:__internal_head__': 1.0, 'memory': 6806163456.0, 'node:172.20.153.209': 1.0, 'object_store_memory': 3403081728.0, 'CPU': 16.0}
INFO flwr 2026-04-21 09:33:56,233 | app.py:219 | Optimize your simulation with Flower VCE: https://flower.dev/docs/framework/how-to-run-simulations.html
INFO flwr 2026-04-21 09:33:56,234 | app.py:242 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 1}
INFO flwr 2026-04-21 09:33:56,250 | app.py:288 | Flower VCE: Creating VirtualClientEngineActorPool with 16 actors
INFO flwr 2026-04-21 09:33:56,251 | server.py:89 | Initializing global parameters
INFO flwr 2026-04-21 09:33:56,252 | server.py:272 | Using initial parameters provided by strategy
INFO

Round 0: centralized test accuracy = 0.5614


DEBUG flwr 2026-04-21 09:34:47,825 | server.py:236 | fit_round 1 received 5 results and 0 failures
WARNING flwr 2026-04-21 09:34:47,829 | fedavg.py:250 | No fit_metrics_aggregation_fn provided
INFO flwr 2026-04-21 09:34:47,833 | server.py:125 | fit progress: (1, 0.2510533118457125, {'accuracy': 0.9473684210526315}, 51.57112239399976)
INFO flwr 2026-04-21 09:34:47,834 | server.py:171 | evaluate_round 1: no clients selected, cancel
DEBUG flwr 2026-04-21 09:34:47,835 | server.py:222 | fit_round 2: strategy sampled 5 clients (out of 5)
DEBUG flwr 2026-04-21 09:34:47,946 | server.py:236 | fit_round 2 received 5 results and 0 failures
INFO flwr 2026-04-21 09:34:47,951 | server.py:125 | fit progress: (2, 0.0897944557823633, {'accuracy': 0.9736842105263158}, 51.689154437999605)
INFO flwr 2026-04-21 09:34:47,952 | server.py:171 | evaluate_round 2: no clients selected, cancel
DEBUG flwr 2026-04-21 09:34:47,953 | server.py:222 | fit_round 3: strategy sampled 5 clients (out of 5)


Round 1: centralized test accuracy = 0.9474
Round 2: centralized test accuracy = 0.9737


DEBUG flwr 2026-04-21 09:34:48,067 | server.py:236 | fit_round 3 received 5 results and 0 failures
INFO flwr 2026-04-21 09:34:48,074 | server.py:125 | fit progress: (3, 0.076728762209154, {'accuracy': 0.9736842105263158}, 51.81149143799939)
INFO flwr 2026-04-21 09:34:48,075 | server.py:171 | evaluate_round 3: no clients selected, cancel
DEBUG flwr 2026-04-21 09:34:48,075 | server.py:222 | fit_round 4: strategy sampled 5 clients (out of 5)
DEBUG flwr 2026-04-21 09:34:48,185 | server.py:236 | fit_round 4 received 5 results and 0 failures
INFO flwr 2026-04-21 09:34:48,191 | server.py:125 | fit progress: (4, 0.07608304486462944, {'accuracy': 0.9649122807017544}, 51.92862289200002)
INFO flwr 2026-04-21 09:34:48,192 | server.py:171 | evaluate_round 4: no clients selected, cancel
DEBUG flwr 2026-04-21 09:34:48,193 | server.py:222 | fit_round 5: strategy sampled 5 clients (out of 5)


...

Round 15: centralized test accuracy = 0.9649
Round 16: centralized test accuracy = 0.9649


DEBUG flwr 2026-04-21 09:34:49,818 | server.py:236 | fit_round 17 received 5 results and 0 failures
INFO flwr 2026-04-21 09:34:49,825 | server.py:125 | fit progress: (17, 0.30519537260042334, {'accuracy': 0.9649122807017544}, 53.5632212699993)
INFO flwr 2026-04-21 09:34:49,826 | server.py:171 | evaluate_round 17: no clients selected, cancel
DEBUG flwr 2026-04-21 09:34:49,827 | server.py:222 | fit_round 18: strategy sampled 5 clients (out of 5)
DEBUG flwr 2026-04-21 09:34:49,934 | server.py:236 | fit_round 18 received 5 results and 0 failures
INFO flwr 2026-04-21 09:34:49,942 | server.py:125 | fit progress: (18, 0.33277751502057445, {'accuracy': 0.9649122807017544}, 53.679316280999956)
INFO flwr 2026-04-21 09:34:49,943 | server.py:171 | evaluate_round 18: no clients selected, cancel
DEBUG flwr 2026-04-21 09:34:49,943 | server.py:222 | fit_round 19: strategy sampled 5 clients (out of 5)


Round 17: centralized test accuracy = 0.9649
Round 18: centralized test accuracy = 0.9649


DEBUG flwr 2026-04-21 09:34:50,077 | server.py:236 | fit_round 19 received 5 results and 0 failures
INFO flwr 2026-04-21 09:34:50,083 | server.py:125 | fit progress: (19, 0.37034241763626796, {'accuracy': 0.9649122807017544}, 53.82084275299985)
INFO flwr 2026-04-21 09:34:50,084 | server.py:171 | evaluate_round 19: no clients selected, cancel
DEBUG flwr 2026-04-21 09:34:50,084 | server.py:222 | fit_round 20: strategy sampled 5 clients (out of 5)
DEBUG flwr 2026-04-21 09:34:50,213 | server.py:236 | fit_round 20 received 5 results and 0 failures
INFO flwr 2026-04-21 09:34:50,218 | server.py:125 | fit progress: (20, 0.3746579719151752, {'accuracy': 0.9649122807017544}, 53.95608343399999)
INFO flwr 2026-04-21 09:34:50,220 | server.py:171 | evaluate_round 20: no clients selected, cancel
INFO flwr 2026-04-21 09:34:50,221 | server.py:153 | FL finished in 53.958287165
INFO flwr 2026-04-21 09:34:50,221 | app.py:226 | app_fit: losses_distributed []
INFO flwr 2026-04-21 09:34:50,222 | app.py:227 |

Round 19: centralized test accuracy = 0.9649
Round 20: centralized test accuracy = 0.9649
Flower simulation completed successfully.


In [87]:
if simulation_ran:
    evaluate_model(server_model, test_loader)
else:
    print("Simulation did not run; no federated model to evaluate.")

Test Accuracy: 0.9649


In [88]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_detailed(model, loader):
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.numpy())
            all_labels.extend(y_batch.numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

In [89]:
print("Centralized Model Performance:")
evaluate_detailed(model, test_loader)

Centralized Model Performance:
Accuracy: 0.9561
Precision: 0.9855
Recall: 0.9444
F1 Score: 0.9645


In [90]:
print("\nFederated Model Performance:")
if simulation_ran:
    evaluate_detailed(server_model, test_loader)
else:
    print("Simulation did not run; metrics unavailable.")


Federated Model Performance:
Accuracy: 0.9649
Precision: 0.9857
Recall: 0.9583
F1 Score: 0.9718
